# GA

Reusable function definitions only. Original project inputs, run settings and saved outputs are not included. Running these cells defines functions; it does not read a dataset. Start with the [synthetic walkthrough](../examples/synthetic_walkthrough.ipynb) for a complete small run.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "helper" / "pipeline_utils.py").is_file()
             and (p / "examples" / "synthetic_demo.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook inside this repository.")
for folder in (ROOT, ROOT / "Invert-PropOnly-saveCSV", ROOT / "Invert-Prop+cond-saveCSV"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


In [ ]:
import random
from typing import List, Tuple

import numpy as np

from Helper_invert_1d import initialize_inversion, fitness_s, unpack_qpso_solutions


def invert_first_output_ga_scaled_Q(
    x_scaler_path: str,
    y_scaler_path: str,
    model_path: str,
    x_fixed_raw: np.ndarray,        # raw fixed inputs (length 10)
    actual_x_adj_raw: np.ndarray,   # raw adjustable inputs (length 4)
    target_raw: float,              # desired raw value of the first output
    pop_size: int,                  # GA population size
    num_generations: int,           # number of GA generations
    crossover_rate: float,          # probability of crossover
    mutation_rate: float,           # probability of mutation per gene
    elitism_count: int,             # number of top individuals to carry over
    hidden_dims: List[int],         # network hidden layer sizes
    activation: str,                # activation used in training
    x_adj_intervals_raw: np.ndarray,# raw [min,max] for each adjustable input (4×2)
    Q_bounds: Tuple[float, float],   # (Q_min, Q_max) in raw space
    k: int                          # number of top unique solutions to return
) -> List[Tuple[np.ndarray, np.ndarray, float]]:
    """
    Genetic‐Algorithm inversion of the first network output under
    x1_raw = Q * x0_raw constraint.

    Returns a list of (x_adj_raw, y_pred_raw, residual) tuples,
    where the first entry is the actual measured input→output.
    """
    # [1] Initialize scalers, model, scaling, actual result, and search bounds
    init = initialize_inversion(
        x_scaler_path,
        y_scaler_path,
        model_path,
        hidden_dims,
        activation,
        x_fixed_raw,
        actual_x_adj_raw,
        target_raw,
        x_adj_intervals_raw,
        Q_bounds
    )
    model         = init["model"]
    x_fixed_s     = init["x_fixed_s"]
    means_x       = init["means_x"]
    scales_x      = init["scales_x"]
    y_scaler      = init["y_scaler"]
    results       = init["results"]        # starts with actual data
    min_vec       = init["min_vec"]        # GA lower bounds
    max_vec       = init["max_vec"]        # GA upper bounds
    target_scaled = init["target_scaled"]  # for fitness

    # [2] Initialize GA population uniformly within bounds
    population = [np.random.uniform(min_vec, max_vec) for _ in range(pop_size)]

    # [3] Evolve population over generations
    for _ in range(num_generations):
        # score each individual
        scored = [
            (fitness_s(ind, model, x_fixed_s, means_x, scales_x, target_scaled), ind)
            for ind in population
        ]
        scored.sort(key=lambda t: t[0])

        # elitism: keep top performers
        new_pop = [ind for (_, ind) in scored[:elitism_count]]

        # fill rest via crossover & mutation
        while len(new_pop) < pop_size:
            # tournament selection
            p1 = min(random.sample(scored, 3), key=lambda t: t[0])[1]
            p2 = min(random.sample(scored, 3), key=lambda t: t[0])[1]

            # crossover
            if random.random() < crossover_rate:
                mask  = np.random.rand(len(min_vec)) < 0.5
                child = np.where(mask, p1, p2)
            else:
                child = p1.copy()

            # mutation
            for j in range(len(min_vec)):
                if random.random() < mutation_rate:
                    child[j] = np.random.uniform(min_vec[j], max_vec[j])

            new_pop.append(child)

        population = new_pop

    # [4] Select top‐k unique solutions based on fitness
    scored_final = [
        (fitness_s(ind, model, x_fixed_s, means_x, scales_x, target_scaled), ind)
        for ind in population
    ]
    scored_final.sort(key=lambda t: t[0])

    unique, seen = [], set()
    for _, cand in scored_final:
        key = tuple(np.round(cand, 6))
        if key in seen:
            continue
        seen.add(key)
        unique.append(cand)
        if len(unique) >= k:
            break

    # [5] Inverse‐scale candidates to raw and compute predictions + residuals
    other_results = unpack_qpso_solutions(
        unique,
        means_x,
        scales_x,
        x_fixed_s,
        model,
        y_scaler,
        target_raw
    )
    results.extend(other_results)

    return results
